# Personnalisation des imports

Ce notebook écrit lui-même les fichiers que ses exercices importent (un module `exports.py`, un dossier `greffons/` et un paquet `rapports/`) dans le répertoire courant : exécutez la cellule suivante d'abord. Les exercices suivent l'ordre du cours : importer par un nom connu à l'exécution, depuis un fichier, au premier accès, puis intervenir dans le mécanisme d'import lui-même.

In [ ]:
from pathlib import Path

Path("greffons").mkdir(exist_ok=True)
Path("rapports").mkdir(exist_ok=True)

LIGNES = [
  {"produit": "stylo", "prix": 1.5},
  {"produit": "cahier", "prix": 3.2},
]

## Charger une classe nommée dans une configuration

Une application d'export lit dans sa configuration la classe à utiliser, sous la forme `"module:Classe"`, comme les points d'entrée d'un `pyproject.toml`. La cellule suivante écrit le module `exports.py`, qui en définit deux.

Écrivez une fonction `charger_objet(reference)` qui importe le module nommé avant le `:` et renvoie l'objet nommé après. Une référence dont le module ou l'objet n'existe pas lève une `ValueError` qui la nomme.

Avec ce code :

```python
for reference in ["exports:ExportCsv", "exports:ExportJson"]:
  print(charger_objet(reference)().exporter(LIGNES))

for reference in ["exports:ExportPdf", "exports_pdf:ExportPdf"]:
  try:
    charger_objet(reference)
  except ValueError as erreur:
    print(erreur)
```

Vous devez obtenir :

```text
produit,prix
stylo,1.5
cahier,3.2
[{"produit": "stylo", "prix": 1.5}, {"produit": "cahier", "prix": 3.2}]
référence inconnue : exports:ExportPdf
référence inconnue : exports_pdf:ExportPdf
```

**Documentation utile**

- [`str.partition`](https://docs.python.org/fr/3/library/stdtypes.html#str.partition)

In [ ]:
%%writefile exports.py
import json


class ExportCsv:
  def exporter(self, lignes):
    colonnes = list(lignes[0])
    resultat = [",".join(colonnes)]
    resultat += [",".join(str(ligne[c]) for c in colonnes) for ligne in lignes]
    return "\n".join(resultat)


class ExportJson:
  def exporter(self, lignes):
    return json.dumps(lignes, ensure_ascii=False)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`"exports:ExportCsv".partition(":")` vaut `("exports", ":", "ExportCsv")`. `importlib.import_module("exports")` renvoie le module (celui de `sys.modules` s'il est déjà importé), et `getattr(module, "ExportCsv")` la classe. Un module introuvable lève `ModuleNotFoundError`, un nom absent du module `AttributeError` : rattrapez les deux, et levez la `ValueError` avec `from erreur` pour garder la cause dans la trace.

```python
import importlib


def charger_objet(reference):
  nom_module, _, nom_objet = reference.partition(":")
  try:
    module = ...
    return ...
  except (ModuleNotFoundError, AttributeError) as erreur:
    raise ValueError(f"référence inconnue : {reference}") from erreur
```

### Solution

In [ ]:
import importlib


def charger_objet(reference):
  nom_module, _, nom_objet = reference.partition(":")
  try:
    module = importlib.import_module(nom_module)
    return getattr(module, nom_objet)
  except (ModuleNotFoundError, AttributeError) as erreur:
    raise ValueError(f"référence inconnue : {reference}") from erreur


for reference in ["exports:ExportCsv", "exports:ExportJson"]:
  print(charger_objet(reference)().exporter(LIGNES))

for reference in ["exports:ExportPdf", "exports_pdf:ExportPdf"]:
  try:
    charger_objet(reference)
  except ValueError as erreur:
    print(erreur)

`ModuleNotFoundError` vient aussi d'un import *à l'intérieur* du module chargé : si `exports.py` importait une bibliothèque non installée, cette solution dirait « référence inconnue » pour un module qui existe. Son attribut `name` dit quel module manque : ne transformez l'erreur que si `erreur.name == nom_module`, et laissez remonter les autres.

## Charger les greffons d'un dossier

Les greffons d'export sont maintenant des fichiers posés dans un dossier `greffons/`, hors de `sys.path`. Chacun définit une constante `EXTENSION` et une fonction `exporter(lignes)`. Les deux cellules suivantes en écrivent deux, `markdown.py` et `html.py`.

1. Écrivez une fonction `charger_greffon(chemin)` qui charge le fichier `chemin` (un `Path`) comme un module, sous le nom `greffon_<nom du fichier sans extension>`, et le renvoie.
2. Écrivez une fonction `charger_greffons(dossier)` qui renvoie un dictionnaire des greffons de tous les fichiers `.py` de `dossier`, indexés par le nom du fichier sans extension.

Avec ce code :

```python
greffons = charger_greffons(Path("greffons"))
print(sorted(greffons))
print(greffons["markdown"].exporter(LIGNES))
print(greffons["html"].EXTENSION, greffons["html"].__name__)
```

Vous devez obtenir :

```text
['html', 'markdown']
| produit | prix |
|---|---|
| stylo | 1.5 |
| cahier | 3.2 |
.html greffon_html
```

**Documentation utile**

- [`pathlib.Path.glob`](https://docs.python.org/fr/3/library/pathlib.html#pathlib.Path.glob)

In [ ]:
%%writefile greffons/markdown.py
EXTENSION = ".md"


def exporter(lignes):
  colonnes = list(lignes[0])
  entete = "| " + " | ".join(colonnes) + " |"
  separateur = "|" + "---|" * len(colonnes)
  corps = [
    "| " + " | ".join(str(ligne[c]) for c in colonnes) + " |" for ligne in lignes
  ]
  return "\n".join([entete, separateur, *corps])

In [ ]:
%%writefile greffons/html.py
EXTENSION = ".html"


def exporter(lignes):
  rangees = [
    "<tr>" + "".join(f"<td>{valeur}</td>" for valeur in ligne.values()) + "</tr>"
    for ligne in lignes
  ]
  return "<table>\n" + "\n".join(rangees) + "\n</table>"

In [ ]:
# Votre code ici

### Conseils supplémentaires

Trois étapes : `importlib.util.spec_from_file_location(nom, chemin)` renvoie la *spec* du fichier (son nom de module, son origine, son chargeur) ; `importlib.util.module_from_spec(spec)` crée un module vide ; `spec.loader.exec_module(module)` exécute le code du fichier dans ce module. Inscrivez le module dans `sys.modules` avant de l'exécuter, comme le fait `import` : le code du greffon qui chercherait son propre module (`dataclasses`, `pickle`) le trouve. `chemin.stem` vaut `"markdown"` pour `greffons/markdown.py`, et `dossier.glob("*.py")` renvoie les fichiers dans un ordre quelconque.

```python
import importlib.util
import sys


def charger_greffon(chemin):
  nom = f"greffon_{chemin.stem}"
  spec = ...
  module = ...
  sys.modules[nom] = module
  ...
  return module


def charger_greffons(dossier):
  return {chemin.stem: ... for chemin in sorted(dossier.glob("*.py"))}
```

### Solution

In [ ]:
import importlib.util
import sys


def charger_greffon(chemin):
  nom = f"greffon_{chemin.stem}"
  spec = importlib.util.spec_from_file_location(nom, chemin)
  module = importlib.util.module_from_spec(spec)
  sys.modules[nom] = module
  spec.loader.exec_module(module)
  return module


def charger_greffons(dossier):
  return {
    chemin.stem: charger_greffon(chemin) for chemin in sorted(dossier.glob("*.py"))
  }


greffons = charger_greffons(Path("greffons"))
print(sorted(greffons))
print(greffons["markdown"].exporter(LIGNES))
print(greffons["html"].EXTENSION, greffons["html"].__name__)

Pourquoi pas `sys.path.insert(0, "greffons")` puis `import_module("html")` ? Le greffon `html.py` porte le nom d'un module de la bibliothèque standard, que Jupyter a déjà importé : `import_module` renverrait ce dernier, sans même lire le fichier. Et s'il ne l'était pas encore, le greffon masquerait le module standard pour tout le programme. Charger le fichier par son chemin, sous un nom préfixé, évite les deux. Un greffon dont le code lève pendant `exec_module` reste inscrit dans `sys.modules`, à moitié initialisé : une application retire son nom (`sys.modules.pop(nom)`) dans un `except` avant de relever l'erreur.

## Un sous-module importé au premier accès

Le paquet `rapports` a deux sous-modules, `pdf` et `tableur`, que les deux cellules suivantes écrivent : chacun affiche un message quand il est importé. On veut que `import rapports` n'importe aucun des deux, et que `rapports.pdf` importe `rapports.pdf` au premier accès seulement.

1. Écrivez le fichier `rapports/__init__.py`, avec `%%writefile rapports/__init__.py` en première ligne de votre cellule : une liste `__all__` des sous-modules, et une fonction `__getattr__(nom)` de module qui importe le sous-module demandé, ou lève l'`AttributeError` habituelle de Python pour un autre nom.
2. Ajoutez une fonction `__dir__()` de module, pour que `dir(rapports)` liste aussi les sous-modules pas encore importés.

Pour vérifier, exécutez ce code dans une nouvelle cellule : ses premières lignes retirent de `sys.modules` le paquet et ses sous-modules, pour que la cellule puisse être relancée après une modification de votre fichier.

```python
import importlib
import sys

for nom in [n for n in sys.modules if n == "rapports" or n.startswith("rapports.")]:
  del sys.modules[nom]
importlib.invalidate_caches()

import rapports

print("pdf" in vars(rapports), "tableur" in dir(rapports))
print(rapports.pdf.generer())
print(rapports.pdf.generer())
print("pdf" in vars(rapports))
try:
  rapports.excel
except AttributeError as erreur:
  print(erreur)
```

Vous devez obtenir :

```text
False True
chargement de rapports.pdf
rapport PDF
rapport PDF
True
module 'rapports' has no attribute 'excel'
```

**Documentation utile**

- [Personnalisation de l'accès aux attributs d'un module](https://docs.python.org/fr/3/reference/datamodel.html#customizing-module-attribute-access)

In [ ]:
%%writefile rapports/pdf.py
print(f"chargement de {__name__}")


def generer():
  return "rapport PDF"

In [ ]:
%%writefile rapports/tableur.py
print(f"chargement de {__name__}")


def generer():
  return "rapport tableur"

In [ ]:
# Votre code ici

### Conseils supplémentaires

Python n'appelle le `__getattr__` d'un module que pour un nom absent du module. `importlib.import_module(f".{nom}", __name__)` importe le sous-module relativement au paquet (`__name__` vaut `"rapports"` dans `__init__.py`), et l'inscrit comme attribut du paquet : au deuxième accès, `__getattr__` n'est plus appelée. Le message habituel est `f"module {__name__!r} has no attribute {nom!r}"`. `__dir__()` ne prend pas d'argument : `globals()` donne les noms déjà définis dans le module.

```python
%%writefile rapports/__init__.py
import importlib

__all__ = ["pdf", "tableur"]


def __getattr__(nom):
  if nom in __all__:
    return ...
  raise AttributeError(...)


def __dir__():
  return sorted(...)
```

### Solution

In [ ]:
%%writefile rapports/__init__.py
import importlib

__all__ = ["pdf", "tableur"]


def __getattr__(nom):
  if nom in __all__:
    return importlib.import_module(f".{nom}", __name__)
  raise AttributeError(f"module {__name__!r} has no attribute {nom!r}")


def __dir__():
  return sorted(set(globals()) | set(__all__))

In [ ]:
import importlib
import sys

for nom in [n for n in sys.modules if n == "rapports" or n.startswith("rapports.")]:
  del sys.modules[nom]
importlib.invalidate_caches()

import rapports

print("pdf" in vars(rapports), "tableur" in dir(rapports))
print(rapports.pdf.generer())
print(rapports.pdf.generer())
print("pdf" in vars(rapports))
try:
  rapports.excel
except AttributeError as erreur:
  print(erreur)

`from rapports import tableur` passe aussi par `__getattr__` : `from` lit l'attribut du paquet, puis, s'il manque, tente d'importer le sous-module. En revanche, `from rapports import *` importe tous les noms de `__all__`, et donc les deux sous-modules : la paresse disparaît.

## Un crochet d'import

Deux chercheurs ajoutés à `sys.meta_path` : le premier sert des modules qui n'existent sur aucun disque, le second (facultatif) refuse certains imports.

### Des modules servis depuis un dictionnaire

Écrivez une classe `ChercheurDictionnaire`, à la fois chercheur (`importlib.abc.MetaPathFinder`) et chargeur (`importlib.abc.Loader`), qui reçoit à sa création un dictionnaire `sources` (nom de module → code source) et sert ces modules à `import`. Une instance de la classe, insérée en tête de `sys.meta_path`, doit rendre importables les modules de `SOURCES`, défini dans la cellule suivante. `exec_module` affiche `exécution de <nom du module>` avant d'exécuter le code.

La cellule doit pouvoir être relancée : avant d'insérer votre chercheur, retirez de `sys.meta_path` ceux d'une exécution précédente, et `unites` de `sys.modules`.

Avec ce code :

```python
import unites
import unites

print(unites.en_cm(10))
```

Vous devez obtenir :

```text
exécution de unites
25.4
```

In [ ]:
SOURCES = {
  "unites": """
CM_PAR_POUCE = 2.54


def en_cm(pouces):
  return pouces * CM_PAR_POUCE
""",
}

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`import unites` interroge les chercheurs de `sys.meta_path` tour à tour, par leur `find_spec(nom, chemin, cible)` : la vôtre renvoie `importlib.util.spec_from_loader(nom, self)` pour un nom de `sources`, et `None` sinon, pour passer la main au chercheur suivant. La machinerie d'import crée alors le module, l'inscrit dans `sys.modules` et appelle `exec_module(module)`, qui exécute le code source avec `exec(code, module.__dict__)`. Le second `import unites` trouve le module dans `sys.modules` : ni `find_spec` ni `exec_module` ne sont appelées, d'où un seul message. Une classe redéfinie à chaque exécution de la cellule n'est plus la même classe : retirez les anciens chercheurs par le nom de leur classe, `type(chercheur).__name__`.

```python
import importlib.abc
import importlib.util
import sys


class ChercheurDictionnaire(importlib.abc.MetaPathFinder, importlib.abc.Loader):
  def __init__(self, sources):
    self.sources = sources

  def find_spec(self, nom, chemin, cible=None):
    if nom not in self.sources:
      return ...
    return ...

  def exec_module(self, module):
    print(f"exécution de {module.__name__}")
    ...


sys.meta_path[:] = [
  chercheur
  for chercheur in sys.meta_path
  if type(chercheur).__name__ != "ChercheurDictionnaire"
]
sys.modules.pop("unites", None)
sys.meta_path.insert(0, ...)
```

#### Solution

In [ ]:
import importlib.abc
import importlib.util
import sys


class ChercheurDictionnaire(importlib.abc.MetaPathFinder, importlib.abc.Loader):
  def __init__(self, sources):
    self.sources = sources

  def find_spec(self, nom, chemin, cible=None):
    if nom not in self.sources:
      return None  # au chercheur suivant de sys.meta_path
    return importlib.util.spec_from_loader(nom, self)

  def exec_module(self, module):
    print(f"exécution de {module.__name__}")
    exec(self.sources[module.__name__], module.__dict__)


# Chercheurs et module d'une exécution précédente de la cellule
sys.meta_path[:] = [
  chercheur
  for chercheur in sys.meta_path
  if type(chercheur).__name__ != "ChercheurDictionnaire"
]
sys.modules.pop("unites", None)
sys.meta_path.insert(0, ChercheurDictionnaire(SOURCES))

import unites
import unites

print(unites.en_cm(10))

### (Facultatif) Refuser un import

Écrivez une classe `ChercheurInterdit`, un chercheur qui reçoit à sa création un ensemble de noms de modules interdits et lève une `ImportError` à l'import de l'un d'eux ou de l'un de leurs sous-modules (`ftplib`, mais aussi `wsgiref.simple_server` si `wsgiref` est interdit). Pour tout autre nom, il passe la main au chercheur suivant. Insérez-en un en tête de `sys.meta_path`, essayez d'importer un module interdit, puis retirez-le de `sys.meta_path`, pour que la suite de la session importe normalement.

Avec `ChercheurInterdit({"ftplib", "wsgiref"})`, ce code :

```python
for nom in ["ftplib", "wsgiref.simple_server", "csv", "json"]:
  try:
    importlib.import_module(nom)
    print(f"{nom} importé")
  except ImportError as erreur:
    print(erreur)
```

doit afficher :

```text
import de ftplib interdit
import de wsgiref interdit
csv importé
json importé
```

Pourquoi `json` s'importe-t-il même si vous l'ajoutez aux noms interdits ?

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Un chercheur qui lève une `ImportError` dans `find_spec` arrête l'import : les chercheurs suivants ne sont pas interrogés. `import wsgiref.simple_server` importe d'abord le paquet `wsgiref` : c'est ce nom que reçoit `find_spec`, d'où le message. Mais si le paquet est déjà chargé, seul `"wsgiref.simple_server"` lui parvient : `nom.partition(".")[0]` en donne `"wsgiref"`. Ce chercheur ne charge rien : il n'a pas besoin d'être un `Loader`. Retirez-le avec `sys.meta_path.remove(chercheur)` dans un `finally`, pour qu'une exception dans la boucle ne le laisse pas en place.

```python
class ChercheurInterdit(importlib.abc.MetaPathFinder):
  def __init__(self, interdits):
    self.interdits = interdits

  def find_spec(self, nom, chemin, cible=None):
    if ...:
      raise ImportError(f"import de {nom} interdit")
    return None


chercheur = ChercheurInterdit({"ftplib", "wsgiref"})
sys.meta_path.insert(0, chercheur)
try:
  ...
finally:
  sys.meta_path.remove(chercheur)
```

#### Solution

In [ ]:
class ChercheurInterdit(importlib.abc.MetaPathFinder):
  def __init__(self, interdits):
    self.interdits = interdits

  def find_spec(self, nom, chemin, cible=None):
    if nom.partition(".")[0] in self.interdits:
      raise ImportError(f"import de {nom} interdit")
    return None


chercheur = ChercheurInterdit({"ftplib", "wsgiref", "json"})
sys.meta_path.insert(0, chercheur)
try:
  for nom in ["ftplib", "wsgiref.simple_server", "csv", "json"]:
    try:
      importlib.import_module(nom)
      print(f"{nom} importé")
    except ImportError as erreur:
      print(erreur)
finally:
  sys.meta_path.remove(chercheur)

`json` est déjà dans `sys.modules` (le notebook l'a importé pour `ExportJson`, et Jupyter avant lui) : `import` le trouve dans le cache, sans interroger aucun chercheur. Un crochet ne voit que les imports d'un module pas encore chargé, et du code peut retirer le crochet de `sys.meta_path` : ce n'est pas un bac à sable. Pour exécuter du code auquel on ne fait pas confiance, il faut l'isoler hors de Python (un conteneur, un processus aux droits réduits).